**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 LangChain 소개 및 기본 실습

## Part 1 | Session 4: LangChain 모듈, LCEL, Memory, 챗봇

---

### 📋 학습 목표

- 🔹 LangChain의 개념과 주요 모듈을 이해합니다
- 🔹 LLM/ChatModel의 기본 사용법을 익힙니다
- 🔹 ChatPromptTemplate으로 프롬프트를 관리합니다
- 🔹 LCEL(LangChain Expression Language)로 파이프라인을 구성합니다
- 🔹 Output Parser로 출력을 구조화합니다
- 🔹 Memory를 활용하여 대화 이력을 관리합니다
- 🔹 간단한 챗봇을 구현합니다

### 📦 필요 라이브러리

```
langchain, langchain-openai, langchain-core, python-dotenv
```

### ⏱️ 예상 소요 시간: 약 90분

---

In [1]:
# 필요 라이브러리 설치
# !pip install langchain langchain-openai langchain-core python-dotenv

import os
from dotenv import load_dotenv

# .env 파일에서 환경변수 로드
load_dotenv()

# API Key 확인
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("✅ OpenAI API Key가 설정되었습니다!")
    print(f"   Key 앞 8자: {api_key[:8]}...")
else:
    print("❌ API Key가 설정되지 않았습니다!")
    print("   .env 파일에 OPENAI_API_KEY=sk-... 를 추가해주세요.")

# LangChain 버전 확인
import langchain
print(f"\n📦 langchain 버전: {langchain.__version__}")

✅ OpenAI API Key가 설정되었습니다!
   Key 앞 8자: sk-proj-...

📦 langchain 버전: 0.3.28


---

## 🎯 LangChain 소개 및 주요 모듈

### LangChain이란?

LangChain은 LLM 기반 어플리케이션을 쉽게 개발할 수 있도록 도와주는 **프레임워크**입니다.

### LangChain의 핵심 가치

- 🔸 **모듈화**: 각 기능을 독립된 모듈로 제공
- 🔸 **조합 가능**: 모듈을 파이프라인으로 연결
- 🔸 **확장 가능**: 다양한 LLM, 벡터DB, 도구 지원
- 🔸 **추상화**: 복잡한 로직을 간단한 인터페이스로

### 주요 모듈 구조

```
LangChain 생태계
  │
  ├── langchain-core       : 핵심 추상화 (LCEL, 메시지, 프롬프트)
  ├── langchain            : 체인, 에이전트, 메모리
  ├── langchain-openai     : OpenAI 통합
  ├── langchain-community  : 커뮤니티 통합
  ├── langgraph            : 에이전트 워크플로우
  └── langsmith            : 디버깅/모니터링
```

### 핵심 구성요소

| 구성요소 | 설명 | 예시 |
|---------|------|------|
| Model | LLM 모델 래퍼 | ChatOpenAI, ChatOllama |
| Prompt | 프롬프트 템플릿 | ChatPromptTemplate |
| Output Parser | 출력 파싱 | StrOutputParser, JsonOutputParser |
| Chain | 모듈 조합 | LCEL 파이프라인 |
| Memory | 대화 이력 관리 | ConversationBufferMemory |
| Retriever | 문서 검색 | VectorStoreRetriever |
| Agent | 도구 활용 자율 행동 | ReAct Agent |

---

---

## 1️⃣ LLM/ChatModel 기본 사용

LangChain에서 LLM 모델을 사용하는 방법을 알아봅니다.

### ChatModel vs LLM

| 구분 | ChatModel | LLM |
|------|-----------|-----|
| 입력 | 메시지 리스트 | 문자열 |
| 출력 | AIMessage 객체 | 문자열 |
| 예시 | ChatOpenAI | OpenAI (deprecated) |

최신 LangChain에서는 **ChatModel**을 사용하는 것이 권장됩니다.

---

In [2]:
# ChatOpenAI 기본 사용
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

print("=" * 60)
print("🤖 ChatOpenAI 기본 사용")
print("=" * 60)

# ChatModel 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=300,
)

print("✅ ChatOpenAI 모델 생성 완료!")
print(f"   모델: {llm.model_name}")
print(f"   Temperature: {llm.temperature}")

# 방법 1: invoke() - 가장 기본적인 호출
print("\n--- 방법 1: invoke() ---")
response = llm.invoke("LangChain이 뭔지 한 문장으로 설명해줘.")
print(f"🤖 응답 타입: {type(response).__name__}")
print(f"🤖 응답: {response.content}")

/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖 ChatOpenAI 기본 사용
✅ ChatOpenAI 모델 생성 완료!
   모델: gpt-4o-mini
   Temperature: 0.7

--- 방법 1: invoke() ---
🤖 응답 타입: AIMessage
🤖 응답: LangChain은 다양한 언어 모델과 데이터 소스를 결합하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.


In [3]:
# 메시지 객체 사용
print("=" * 60)
print("📨 메시지 객체 사용")
print("=" * 60)

# 방법 2: 메시지 리스트로 호출
messages = [
    SystemMessage(content="당신은 파이썬 전문가입니다. 한국어로 간결하게 답변하세요."),
    HumanMessage(content="리스트 컴프리헨션이 뭐야?"),
]

response = llm.invoke(messages)
print(f"\n🤖 응답:\n{response.content}")

# 응답 메타데이터
print(f"\n📊 메타데이터:")
print(f"   토큰 사용: {response.usage_metadata}")
print(f"   응답 ID: {response.id}")

📨 메시지 객체 사용

🤖 응답:
리스트 컴프리헨션(List Comprehension)은 파이썬에서 리스트를 간결하게 생성하는 방법입니다. 기존 리스트나 iterable을 기반으로 새로운 리스트를 만들 수 있으며, 조건문도 포함할 수 있습니다.

예를 들어, 0부터 9까지의 제곱을 구하는 리스트는 다음과 같이 만들 수 있습니다:

```python
squares = [x ** 2 for x in range(10)]
```

조건을 추가할 수도 있습니다:

```python
even_squares = [x ** 2 for x in range(10) if x % 2 == 0]
```

이렇게 하면 짝수의 제곱만 포함된 리스트가 생성됩니다. 리스트 컴프리헨션은 코드가 간결하고 읽기 쉽게 만들어 줍니다.

📊 메타데이터:
   토큰 사용: {'input_tokens': 43, 'output_tokens': 172, 'total_tokens': 215, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
   응답 ID: run--019db8e8-fb3a-7de3-badb-a5dada6104fb-0


In [4]:
# 스트리밍 사용
print("=" * 60)
print("🌊 LangChain 스트리밍")
print("=" * 60)

print("\n🤖 스트리밍 응답:")
print("-" * 40)

# stream() 메서드로 스트리밍
for chunk in llm.stream("파인튜닝의 장점을 3가지 알려줘."):
    print(chunk.content, end="", flush=True)

print("\n" + "-" * 40)
print("\n✅ stream() 메서드로 간편하게 스트리밍 가능!")

🌊 LangChain 스트리밍

🤖 스트리밍 응답:
----------------------------------------
파인튜닝(fine-tuning)은 사전 훈련된 모델을 특정 작업이나 데이터셋에 맞게 조정하는 과정으로, 여러 가지 장점이 있습니다. 그 중 세 가지를 소개하겠습니다.

1. **효율적인 자원 활용**: 파인튜닝은 대규모 데이터셋에서 이미 학습된 모델을 기반으로 하므로, 처음부터 모델을 훈련시키는 것보다 훨씬 적은 데이터와 계산 자원으로도 높은 성능을 낼 수 있습니다. 이를 통해 시간과 비용을 절약할 수 있습니다.

2. **특정 도메인에 대한 성능 향상**: 파인튜닝을 통해 특정 도메인이나 작업에 맞게 모델을 최적화할 수 있습니다. 이를 통해 해당 분야에서의 성능을 극대화할 수 있으며, 일반적인 모델보다 더 정확한 결과를 얻을 수 있습니다.

3. **전이 학습의 이점**: 파인튜닝은 전이 학습(transfer learning)의 일환으로, 기존의 지식을 새로운 작업에 효과적으로 전이할 수 있게 해줍니다. 이는 데이터가 부족한 상황에서도 모델의 성능을 높일 수 있는 강력한 방법입니다. 사전 훈련된 모델이 일반적인 패턴을 학습했기 때문에, 이러한 지식을 활용하여 새로운 작업에 빠르게 적응할 수 있습니다
----------------------------------------

✅ stream() 메서드로 간편하게 스트리밍 가능!


---

## 2️⃣ 프롬프트 템플릿 (ChatPromptTemplate)

프롬프트를 **재사용 가능한 템플릿**으로 관리합니다.

### 왜 프롬프트 템플릿이 필요한가?

- 🔸 프롬프트의 **재사용성** 향상
- 🔸 **변수 치환**으로 동적 프롬프트 생성
- 🔸 프롬프트 **버전 관리** 용이
- 🔸 **일관된 형식** 유지

---

In [5]:
# ChatPromptTemplate 기본 사용
from langchain_core.prompts import ChatPromptTemplate

print("=" * 60)
print("📝 ChatPromptTemplate 기본 사용")
print("=" * 60)

# 방법 1: from_messages로 생성
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다. {language}로 답변하세요."),
    ("human", "{question}"),
])

print("📌 템플릿 변수:")
print(f"   {prompt.input_variables}")

# 변수 치환하여 메시지 생성
messages = prompt.format_messages(
    role="머신러닝",
    language="한국어",
    question="오버피팅을 방지하는 방법은?"
)

print(f"\n📨 생성된 메시지:")
for msg in messages:
    print(f"   [{msg.type}] {msg.content}")

# LLM 호출
response = llm.invoke(messages)
print(f"\n🤖 응답:\n{response.content}")

📝 ChatPromptTemplate 기본 사용
📌 템플릿 변수:
   ['language', 'question', 'role']

📨 생성된 메시지:
   [system] 당신은 머신러닝 전문가입니다. 한국어로 답변하세요.
   [human] 오버피팅을 방지하는 방법은?

🤖 응답:
오버피팅을 방지하는 방법에는 여러 가지가 있습니다. 다음은 일반적으로 사용되는 몇 가지 기법입니다:

1. **훈련 데이터 증가**: 더 많은 데이터를 수집하거나 데이터 증강 기법을 사용하여 훈련 데이터를 늘립니다. 이는 모델이 다양한 패턴을 학습하도록 도와줍니다.

2. **정규화**: L1, L2 정규화와 같은 기법을 사용하여 모델의 복잡성을 줄입니다. 이러한 기법은 가중치에 패널티를 부여하여 과도한 학습을 방지합니다.

3. **드롭아웃**: 신경망에서는 드롭아웃(Dropout) 기법을 사용하여 훈련 중 일부 뉴런을 임의로 제거함으로써 모델의 일반화 능력을 향상시킵니다.

4. **조기 종료 (Early Stopping)**: 훈련 중 검증 데이터의 성능이 저하되기 시작하면 훈련을 중단하는 기법입니다. 이를 통해 모델이 과적합되는 것을 방지할 수 있습니다.

5. **모델 단순화**: 불필요한 복잡성을 줄이기 위해 모델의 구조를 단순화합니다. 예를 들어, 층의 수를 줄이거나 노드 수를 감소시킬 수 있습니다.

6. **교차 검증**: 교


In [6]:
# 다양한 프롬프트 템플릿 예시
print("=" * 60)
print("📝 실용적인 프롬프트 템플릿 예시")
print("=" * 60)

# 번역 템플릿
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "{source_lang}를 {target_lang}로 번역하세요. 번역 결과만 출력하세요."),
    ("human", "{text}"),
])

# 요약 템플릿
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트를 {num_sentences}문장으로 요약하세요."),
    ("human", "{text}"),
])

# 번역 실행
print("\n--- 번역 ---")
msg = translate_prompt.format_messages(
    source_lang="한국어",
    target_lang="영어",
    text="파인튜닝은 사전 학습된 모델을 특정 작업에 맞게 추가 학습하는 기법입니다."
)
response = llm.invoke(msg)
print(f"🤖 {response.content}")

# 요약 실행
print("\n--- 요약 ---")
msg = summary_prompt.format_messages(
    num_sentences=2,
    text="""트랜스포머는 2017년 구글에서 발표한 신경망 아키텍처입니다. 
    기존 RNN/LSTM의 순차적 처리 한계를 극복하고, 셀프 어텐션 메커니즘을 통해 
    입력 시퀀스의 모든 위치를 동시에 참조할 수 있습니다. 이를 통해 병렬 처리가 가능해져 
    학습 속도가 크게 향상되었으며, GPT, BERT, T5 등 현대 LLM의 기반이 되었습니다."""
)
response = llm.invoke(msg)
print(f"🤖 {response.content}")

print("\n✅ 템플릿을 미리 정의하면 일관된 프롬프트를 재사용할 수 있습니다!")

📝 실용적인 프롬프트 템플릿 예시

--- 번역 ---
🤖 Fine-tuning is a technique for further training a pre-trained model to adapt it to a specific task.

--- 요약 ---
🤖 트랜스포머는 2017년 구글에서 발표된 신경망 아키텍처로, RNN/LSTM의 한계를 극복하고 셀프 어텐션 메커니즘을 통해 입력 시퀀스를 동시에 처리할 수 있게 해줍니다. 이로 인해 학습 속도가 향상되었으며, GPT, BERT, T5 등의 현대 대형 언어 모델의 기반이 됩니다.

✅ 템플릿을 미리 정의하면 일관된 프롬프트를 재사용할 수 있습니다!


---

## 3️⃣ LCEL (LangChain Expression Language) 파이프라인

LCEL은 LangChain의 **파이프라인 구성 문법**입니다. `|` (파이프) 연산자로 모듈을 연결합니다.

### LCEL의 핵심 개념

```python
chain = prompt | model | output_parser
#       입력     →    처리     →    출력 파싱
```

### LCEL의 장점

- 🔸 **직관적 구문**: `|`로 데이터 흐름을 표현
- 🔸 **자동 스트리밍**: chain.stream()으로 자동 지원
- 🔸 **비동기 지원**: chain.ainvoke()로 비동기 실행
- 🔸 **배치 처리**: chain.batch()로 대량 처리
- 🔸 **재시도/폴백**: with_retry(), with_fallbacks()

---

In [7]:
# LCEL 기본 파이프라인
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

print("=" * 60)
print("🔗 LCEL 기본 파이프라인")
print("=" * 60)

# 파이프라인 구성: 프롬프트 → 모델 → 출력 파서
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {topic} 전문가입니다. 한국어로 간결하게 답변하세요."),
    ("human", "{question}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
output_parser = StrOutputParser()

# LCEL 체인 생성 ( | 연산자로 연결)
chain = prompt | model | output_parser

print("✅ LCEL 체인 생성 완료!")
print(f"   체인 구조: prompt → model → output_parser")

# invoke() 호출
result = chain.invoke({
    "topic": "딥러닝",
    "question": "CNN과 RNN의 차이점을 설명해주세요."
})

print(f"\n📝 입력: topic='딥러닝', question='CNN과 RNN의 차이점을 설명해주세요.'")
print(f"\n🤖 출력 (문자열):")
print(result)
print(f"\n📊 출력 타입: {type(result).__name__}")
print("💡 StrOutputParser가 AIMessage를 문자열로 변환했습니다!")

🔗 LCEL 기본 파이프라인
✅ LCEL 체인 생성 완료!
   체인 구조: prompt → model → output_parser

📝 입력: topic='딥러닝', question='CNN과 RNN의 차이점을 설명해주세요.'

🤖 출력 (문자열):
CNN(합성곱 신경망)과 RNN(순환 신경망)은 각각 다른 유형의 데이터를 처리하는 데 최적화된 신경망입니다.

1. **구조**:
   - CNN: 주로 이미지 데이터를 처리하며, 합성곱층과 풀링층으로 구성되어 있습니다. 지역적인 특징 추출에 강합니다.
   - RNN: 시계열 데이터나 순차적인 데이터를 처리하는 데 적합하며, 이전 단계의 출력을 다음 단계의 입력으로 사용하는 순환 구조를 가지고 있습니다.

2. **데이터 처리 방식**:
   - CNN: 고정된 크기의 입력을 처리하고, 공간적인 정보를 보존하며 특징 맵을 생성합니다.
   - RNN: 가변 길이의 입력을 처리할 수 있으며, 시간적인 의존성을 모델링할 수 있습니다.

3. **주요 용도**:
   - CNN: 이미지 분류, 객체 탐지, 이미지 생성 등.
   - RNN: 자연어 처리, 음성 인식, 시계열 예측 등.

이러한 차이로 인해 각각의 네트워크는 특정 문제에 더 적합합니다.

📊 출력 타입: str
💡 StrOutputParser가 AIMessage를 문자열로 변환했습니다!


In [8]:
# LCEL 스트리밍과 배치 처리
print("=" * 60)
print("🔗 LCEL 스트리밍 & 배치 처리")
print("=" * 60)

# 스트리밍
print("\n--- 스트리밍 ---")
print("🤖 ", end="")
for chunk in chain.stream({"topic": "파이썬", "question": "데코레이터를 간단히 설명해줘."}):
    print(chunk, end="", flush=True)
print()

# 배치 처리
print("\n--- 배치 처리 ---")
questions = [
    {"topic": "파이썬", "question": "GIL이 뭐야?"},
    {"topic": "딥러닝", "question": "드롭아웃이 뭐야?"},
    {"topic": "데이터", "question": "정규화가 뭐야?"},
]

results = chain.batch(questions)

for q, r in zip(questions, results):
    print(f"\n📝 [{q['topic']}] {q['question']}")
    print(f"🤖 {r[:100]}...")

print(f"\n✅ batch()로 {len(questions)}개의 질문을 한 번에 처리했습니다!")

🔗 LCEL 스트리밍 & 배치 처리

--- 스트리밍 ---
🤖 데코레이터는 함수나 메서드를 수정하거나 확장하는 데 사용되는 Python의 기능입니다. 주로 함수의 실행 전후에 추가적인 기능을 넣고 싶을 때 사용합니다. 데코레이터는 다른 함수를 인자로 받고, 새로운 함수를 반환하는 형태로 작동합니다.

예를 들어:

```python
def my_decorator(func):
    def wrapper():
        print("Before the function call.")
        func()
        print("After the function call.")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
```

위 코드에서 `say_hello` 함수는 `my_decorator`에 의해 감싸져 추가적인 기능이 적용됩니다.

--- 배치 처리 ---

📝 [파이썬] GIL이 뭐야?
🤖 GIL(전역 인터프리터 잠금)은 CPython 인터프리터에서 여러 스레드가 동시에 실행될 때, 한 번에 하나의 스레드만 Python 바이트코드를 실행할 수 있도록 하는 잠금입니다....

📝 [딥러닝] 드롭아웃이 뭐야?
🤖 드롭아웃(Dropout)은 딥러닝 모델의 과적합(overfitting)을 방지하기 위한 정규화 기법입니다. 학습 과정에서 무작위로 일정 비율의 뉴런을 활성화하지 않고 '드롭'하여,...

📝 [데이터] 정규화가 뭐야?
🤖 정규화는 데이터베이스 설계에서 데이터 중복을 줄이고 무결성을 높이기 위한 과정입니다. 이를 통해 데이터를 여러 개의 테이블로 나누고, 각 테이블 간의 관계를 정의하여 효율적인 데이...

✅ batch()로 3개의 질문을 한 번에 처리했습니다!


In [9]:
# ainvoke vs batch 비교
import asyncio
import time

print("=" * 60)
print("⚡ ainvoke vs batch 비교")
print("=" * 60)

questions = [
    {"topic": "파이썬", "question": "GIL이 뭐야?"},
    {"topic": "딥러닝", "question": "드롭아웃이 뭐야?"},
    {"topic": "데이터", "question": "정규화가 뭐야?"},
]

# ===== 1. invoke 순차 처리 =====
print("\n1️⃣ invoke() — 순차 처리")
start = time.time()
results_sync = []
for q in questions:
    results_sync.append(chain.invoke(q))
sync_time = time.time() - start
print(f"   ⏱️ {sync_time:.2f}초")

# ===== 2. batch 동시 처리 =====
print("\n2️⃣ batch() — 스레드 동시 처리")
start = time.time()
results_batch = chain.batch(questions)
batch_time = time.time() - start
print(f"   ⏱️ {batch_time:.2f}초")

# ===== 3. ainvoke 비동기 동시 처리 =====
print("\n3️⃣ ainvoke() — 비동기 동시 처리")

async def run_async():
    start = time.time()
    results = await asyncio.gather(
        chain.ainvoke(questions[0]),
        chain.ainvoke(questions[1]),
        chain.ainvoke(questions[2]),
    )
    elapsed = time.time() - start
    return results, elapsed

results_async, async_time = await run_async()

# ===== 결과 비교 =====
print(f"   ⏱️ {async_time:.2f}초")
print("\n" + "=" * 60)
print("📊 시간 비교")
print("=" * 60)
print(f"   invoke  (순차):  {sync_time:.2f}초")
print(f"   batch   (스레드): {batch_time:.2f}초")
print(f"   ainvoke (비동기): {async_time:.2f}초")
print(f"\n💡 batch와 ainvoke는 비슷한 속도, invoke 순차 처리는 느림!")
print("💡 Jupyter는 이미 이벤트 루프가 있어서 await를 바로 사용할 수 있습니다.")

⚡ ainvoke vs batch 비교

1️⃣ invoke() — 순차 처리
   ⏱️ 6.53초

2️⃣ batch() — 스레드 동시 처리
   ⏱️ 2.56초

3️⃣ ainvoke() — 비동기 동시 처리
   ⏱️ 2.45초

📊 시간 비교
   invoke  (순차):  6.53초
   batch   (스레드): 2.56초
   ainvoke (비동기): 2.45초

💡 batch와 ainvoke는 비슷한 속도, invoke 순차 처리는 느림!
💡 Jupyter는 이미 이벤트 루프가 있어서 await를 바로 사용할 수 있습니다.


In [10]:
# 체인 연결: 다단계 파이프라인
print("=" * 60)
print("🔗 다단계 LCEL 파이프라인")
print("=" * 60)

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 1단계: 질문에 대한 답변 생성
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 질문에 한국어로 상세히 답변하세요."),
    ("human", "{question}"),
])

# 2단계: 답변을 요약
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트를 핵심만 2줄로 요약하세요."),
    ("human", "{text}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
parser = StrOutputParser()

# 1단계 체인
answer_chain = answer_prompt | model | parser

# 2단계 체인 (1단계 결과를 입력으로)
full_chain = (
    answer_chain  # 1단계: 답변 생성
    | RunnableLambda(lambda x: {"text": x})  # 결과를 dict로 변환
    | summary_prompt  # 2단계: 요약 프롬프트
    | model  # 2단계: 모델 호출
    | parser  # 2단계: 문자열 파싱
)

# 실행
question = "트랜스포머의 셀프 어텐션 메커니즘이 뭐야?"
print(f"\n📝 질문: {question}")

# 1단계 결과
print(f"\n--- 1단계: 상세 답변 ---")
answer = answer_chain.invoke({"question": question})
print(f"🤖 {answer}")

# 2단계 결과 (전체 파이프라인)
print(f"\n--- 2단계: 요약 ---")
summary = full_chain.invoke({"question": question})
print(f"🤖 {summary}")

print("\n✅ 다단계 파이프라인으로 '답변 생성 → 요약'을 자동화했습니다!")

🔗 다단계 LCEL 파이프라인

📝 질문: 트랜스포머의 셀프 어텐션 메커니즘이 뭐야?

--- 1단계: 상세 답변 ---
🤖 트랜스포머의 셀프 어텐션 메커니즘은 자연어 처리(NLP) 분야에서 매우 중요한 역할을 하는 기술입니다. 이 메커니즘은 입력된 문장이나 데이터의 각 요소가 서로 어떻게 관계를 맺고 있는지를 파악하는 데 도움을 줍니다. 셀프 어텐션은 특히 문맥을 이해하는 데 유용합니다. 다음은 셀프 어텐션 메커니즘의 작동 방식에 대한 상세한 설명입니다.

### 1. 입력 표현
트랜스포머 모델은 입력 데이터를 단어의 임베딩 벡터로 변환하여 처리합니다. 각 단어는 고차원 공간에서 벡터로 표현되며, 이 벡터들은 문맥 정보를 포함하고 있습니다.

### 2. 쿼리, 키, 밸류 생성
셀프 어텐션에서는 각 단어 벡터로부터 세 가지 벡터를 생성합니다:
- **쿼리 (Query)**: 현재 단어가 다른 단어에 대해 얼마나 주목해야 하는지를 나타내는 벡터.
- **키 (Key)**: 각 단어의 정보가 얼마나 중요한지를 나타내는 벡터.
- **밸류 (Value)**: 실제로 참조할 정보가 담긴 벡터.

이 세 가지는 입력 벡터에 대해 학습된 가중치 행렬을 곱하여 생성됩니다.

### 3. 어텐션 점수 계산
각 단어의 쿼리 벡터와 다른 모든 단어의 키 벡터 간의 내적을 계산하여 어텐션 점수를 구합니다. 이 점수는 현재 단어가 다른 단어에 얼마나 집중해야 하는지를 나타냅니다. 이 값이 높을수록 해당 단어에 더 많은 주의를 기울여야 한다는 의미입니다.

### 4. 소프트맥스 적용
어텐션 점수에 소프트맥스 함수를 적용하여 각 단어에 대한 확률 분포를 만듭니다. 이 확률은 각 단어가 얼마나 중요한지를 나타내며, 모든 점수의 합은 1이 됩니다.

### 5. 가중합 계산
이제 각 단어의 밸류 벡터에 소프트맥스 결과로 얻은 확률을 곱하여 가중합을 계산합니다. 이 과정은 각 단어가 다른 단어로부터 얼마나 많은 정보를 가져오는지를 결정합니다.

### 6. 최종 출력
이러한 가중합 결과

---

## 4️⃣ Output Parser 활용

Output Parser는 LLM 출력을 **원하는 형태**로 변환합니다.

### 주요 Output Parser

| Parser | 설명 | 출력 타입 |
|--------|------|----------|
| StrOutputParser | 문자열로 변환 | str |
| JsonOutputParser | JSON으로 파싱 | dict |
| CommaSeparatedListOutputParser | 쉼표 구분 리스트 | list |
| PydanticOutputParser | Pydantic 모델로 파싱 | Pydantic 객체 |

---

In [11]:
# 다양한 Output Parser 활용
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser

print("=" * 60)
print("📤 Output Parser 활용")
print("=" * 60)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. CommaSeparatedListOutputParser
print("\n--- 1. CommaSeparatedListOutputParser ---")
list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "답변을 쉼표로 구분된 리스트로 작성하세요. {format_instructions}"),
    ("human", "{question}"),
])

list_chain = list_prompt | model | list_parser

result = list_chain.invoke({
    "question": "인기 있는 딥러닝 프레임워크 5가지를 알려줘",
    "format_instructions": list_parser.get_format_instructions(),
})

print(f"📊 결과 타입: {type(result).__name__}")
print(f"📊 결과: {result}")
print(f"📊 첫 번째 항목: {result[0]}")

# 2. JsonOutputParser
print("\n--- 2. JsonOutputParser ---")
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 형식의 JSON으로 응답하세요:
{{"name": "모델명", "organization": "개발사", "parameters": "파라미터수", "year": 연도}}
{format_instructions}"""),
    ("human", "{question}"),
])

json_chain = json_prompt | model | json_parser

result = json_chain.invoke({
    "question": "GPT-4에 대해 알려줘",
    "format_instructions": json_parser.get_format_instructions(),
})

print(f"📊 결과 타입: {type(result).__name__}")
print(f"📊 결과: {result}")
print(f"📊 모델명: {result.get('name')}")

print("\n✅ Output Parser로 LLM 출력을 프로그래밍에 바로 활용할 수 있습니다!")

📤 Output Parser 활용

--- 1. CommaSeparatedListOutputParser ---
📊 결과 타입: list
📊 결과: ['TensorFlow', 'PyTorch', 'Keras', 'MXNet', 'Caffe']
📊 첫 번째 항목: TensorFlow

--- 2. JsonOutputParser ---
📊 결과 타입: dict
📊 결과: {'name': 'GPT-4', 'organization': 'OpenAI', 'parameters': '수천억', 'year': 2023}
📊 모델명: GPT-4

✅ Output Parser로 LLM 출력을 프로그래밍에 바로 활용할 수 있습니다!


---

## 5️⃣ Memory와 대화 이력 관리

LLM API는 기본적으로 **상태를 저장하지 않습니다(stateless)**.

LangChain의 Memory 모듈을 사용하면 **대화 이력을 자동으로 관리**할 수 있습니다.

### 주요 Memory 유형

| Memory | 설명 | 사용 사례 |
|--------|------|----------|
| ConversationBufferMemory | 전체 대화 저장 | 짧은 대화 |
| ConversationBufferWindowMemory | 최근 N턴만 저장 | 일반적인 챗봇 |
| ConversationSummaryMemory | 대화를 요약하여 저장 | 긴 대화 |
| ConversationTokenBufferMemory | 토큰 수 기반 관리 | 비용 최적화 |

### 최신 LangChain 방식: 메시지 히스토리 직접 관리

최신 LangChain에서는 `RunnableWithMessageHistory`나 직접 메시지 리스트를 관리하는 방식이 권장됩니다.

---

In [12]:
# 메시지 히스토리 직접 관리 방식 (최신 LangChain 권장)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("=" * 60)
print("💾 메시지 히스토리를 활용한 대화 관리")
print("=" * 60)

# MessagesPlaceholder를 사용한 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요."),
    MessagesPlaceholder(variable_name="history"),  # 대화 이력이 여기에 삽입
    ("human", "{input}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()
chain = prompt | model | parser

# 대화 이력 저장소
chat_history = []

def chat_with_history(user_input):
    """대화 이력을 관리하면서 채팅"""
    # 체인 실행
    response = chain.invoke({
        "history": chat_history,
        "input": user_input,
    })
    
    # 대화 이력에 추가
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))
    
    return response

# 대화 진행
print("\n👤 1턴: 나는 파이썬을 공부하고 있어.")
print(f"🤖 {chat_with_history('나는 파이썬을 공부하고 있어.')}")

print("\n" + "-" * 40)
print("\n👤 2턴: 어떤 프레임워크를 배우면 좋을까?")
print(f"🤖 {chat_with_history('어떤 프레임워크를 배우면 좋을까?')}")

print("\n" + "-" * 40)
print("\n👤 3턴: 내가 지금 뭘 공부하고 있다고 했지?")
print(f"🤖 {chat_with_history('내가 지금 뭘 공부하고 있다고 했지?')}")

print(f"\n📊 대화 이력: {len(chat_history)}개 메시지")
print("✅ 이전 대화 맥락을 기억하고 있습니다!")

💾 메시지 히스토리를 활용한 대화 관리

👤 1턴: 나는 파이썬을 공부하고 있어.
🤖 좋아요! 파이썬은 배우기 쉽고 다양한 분야에서 사용되는 프로그래밍 언어입니다. 어떤 부분을 공부하고 있나요? 기본 문법, 데이터 구조, 웹 개발, 데이터 분석 등 다양한 주제가 있을 수 있습니다. 도움이 필요하면 언제든지 질문해 주세요!

----------------------------------------

👤 2턴: 어떤 프레임워크를 배우면 좋을까?
🤖 파이썬에는 여러 유용한 프레임워크가 있어서 목적에 따라 선택할 수 있습니다. 다음은 몇 가지 추천하는 프레임워크입니다:

1. **웹 개발**:
   - **Django**: 강력한 웹 프레임워크로, 많은 기능이 내장되어 있어 대규모 애플리케이션 개발에 적합합니다.
   - **Flask**: 경량 웹 프레임워크로, 간단한 애플리케이션을 빠르게 만들고 싶을 때 좋습니다.

2. **데이터 과학 및 머신러닝**:
   - **Pandas**: 데이터 분석과 조작을 위한 라이브러리입니다.
   - **NumPy**: 수치 계산을 위한 라이브러리로, 배열 및 행렬 연산에 유용합니다.
   - **TensorFlow** 또는 **PyTorch**: 머신러닝 및 딥러닝을 위한 프레임워크입니다.

3. **자동화 및 스크립팅**:
   - **Selenium**: 웹 브라우저 자동화를 위한 도구로, 웹 애플리케이션 테스트에 사용됩니다.
   - **Beautiful Soup**: 웹 스크래핑을 위한 라이브러리로, HTML/XML 문서에서 데이터를 추출하는 데 유용합니다.

어떤 분야에 관심이 있는지에 따라 선택할 수 있습니다. 추가적인 조언이 필요하면 말씀해 주세요!

----------------------------------------

👤 3턴: 내가 지금 뭘 공부하고 있다고 했지?
🤖 당신은 현재 파이썬을 공부하고 있다고 말씀하셨습니다. 파이썬과 관련된 어떤 주제나 프레임워크에 대해 더 알고 싶으신지 말씀해 주시면, 더 

In [13]:
# 대화 이력 확인 및 관리 전략
print("=" * 60)
print("📋 대화 이력 확인")
print("=" * 60)

for i, msg in enumerate(chat_history):
    role = "👤 User" if isinstance(msg, HumanMessage) else "🤖 AI"
    content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"\n[{i}] {role}: {content}")

print(f"\n{'=' * 60}")
print("💡 대화 이력 관리 전략:")
print("   1. 전체 보관: 짧은 대화에 적합")
print("   2. 윈도우: 최근 N턴만 유지 (메모리/비용 절약)")
print("   3. 요약: 오래된 대화를 요약하여 보관")
print("   4. 토큰 제한: 최대 토큰 수 기준으로 관리")

# 윈도우 방식 예시
MAX_HISTORY = 4  # 최근 4개 메시지만 유지 (2턴)
if len(chat_history) > MAX_HISTORY:
    trimmed = chat_history[-MAX_HISTORY:]
    print(f"\n📌 윈도우 적용 예시: {len(chat_history)}개 → {len(trimmed)}개")

📋 대화 이력 확인

[0] 👤 User: 나는 파이썬을 공부하고 있어.

[1] 🤖 AI: 좋아요! 파이썬은 배우기 쉽고 다양한 분야에서 사용되는 프로그래밍 언어입니다. 어떤 부분을 공부하고 있나요? 기본 문법, 데이터 구조, 웹 개발...

[2] 👤 User: 어떤 프레임워크를 배우면 좋을까?

[3] 🤖 AI: 파이썬에는 여러 유용한 프레임워크가 있어서 목적에 따라 선택할 수 있습니다. 다음은 몇 가지 추천하는 프레임워크입니다:

1. **웹 개발**:...

[4] 👤 User: 내가 지금 뭘 공부하고 있다고 했지?

[5] 🤖 AI: 당신은 현재 파이썬을 공부하고 있다고 말씀하셨습니다. 파이썬과 관련된 어떤 주제나 프레임워크에 대해 더 알고 싶으신지 말씀해 주시면, 더 구체적...

💡 대화 이력 관리 전략:
   1. 전체 보관: 짧은 대화에 적합
   2. 윈도우: 최근 N턴만 유지 (메모리/비용 절약)
   3. 요약: 오래된 대화를 요약하여 보관
   4. 토큰 제한: 최대 토큰 수 기준으로 관리

📌 윈도우 적용 예시: 6개 → 4개


---

## 6️⃣ 간단한 챗봇 구현

지금까지 배운 내용을 종합하여 **대화형 챗봇**을 구현합니다.

### 챗봇 구성요소

1. **시스템 프롬프트**: 챗봇의 역할 정의
2. **프롬프트 템플릿**: MessagesPlaceholder로 이력 관리
3. **LCEL 체인**: prompt | model | parser
4. **대화 이력**: 최근 N턴 윈도우 관리

---

In [14]:
# 간단한 챗봇 클래스 구현
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI

print("=" * 60)
print("🤖 간단한 챗봇 구현")
print("=" * 60)

class SimpleChatbot:
    """LangChain 기반 간단한 챗봇"""
    
    def __init__(self, system_prompt, model_name="gpt-4o-mini", 
                 temperature=0.7, max_history=10):
        self.history = []
        self.max_history = max_history
        
        # 프롬프트 템플릿
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ])
        
        # 모델
        self.model = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            max_tokens=500,
        )
        
        # 체인 구성
        self.chain = self.prompt | self.model | StrOutputParser()
        
        print(f"✅ 챗봇 초기화 완료!")
        print(f"   모델: {model_name}")
        print(f"   최대 이력: {max_history}개 메시지")
    
    def chat(self, user_input):
        """사용자 입력에 대한 응답 생성"""
        # 체인 실행
        response = self.chain.invoke({
            "history": self.history,
            "input": user_input,
        })
        
        # 이력 추가
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=response))
        
        # 이력 크기 관리 (윈도우)
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
        
        return response
    
    def stream_chat(self, user_input):
        """스트리밍 응답 생성"""
        full_response = ""
        for chunk in self.chain.stream({"history": self.history, "input": user_input}):
            full_response += chunk
            yield chunk
        
        # 이력 추가
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=full_response))
        
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
    
    def reset(self):
        """대화 이력 초기화"""
        self.history = []
        print("🔄 대화 이력이 초기화되었습니다.")
    
    def get_history_count(self):
        """현재 대화 이력 수"""
        return len(self.history)

# 챗봇 생성
bot = SimpleChatbot(
    system_prompt="""당신은 AI/ML 학습을 도와주는 친절한 튜터입니다.
규칙:
- 한국어로 답변합니다.
- 초보자도 이해할 수 있게 쉽게 설명합니다.
- 필요하면 예시 코드를 포함합니다.
- 답변은 간결하게 유지합니다.""",
    max_history=10,
)

🤖 간단한 챗봇 구현
✅ 챗봇 초기화 완료!
   모델: gpt-4o-mini
   최대 이력: 10개 메시지


In [15]:
# 챗봇과 대화하기
print("=" * 60)
print("💬 챗봇과 대화하기")
print("=" * 60)

# 대화 1
print("\n👤 User: LoRA가 뭐야?")
print(f"🤖 Bot: {bot.chat('LoRA가 뭐야?')}")

# 대화 2
print(f"\n{'─' * 50}")
print("\n👤 User: 그거랑 QLoRA는 뭐가 달라?")
print(f"🤖 Bot: {bot.chat('그거랑 QLoRA는 뭐가 달라?')}")

# 대화 3 (스트리밍)
print(f"\n{'─' * 50}")
print("\n👤 User: 내 GPU가 8GB인데 어떤 모델까지 학습 가능해?")
print("🤖 Bot: ", end="")
for chunk in bot.stream_chat("내 GPU가 8GB인데 어떤 모델까지 학습 가능해?"):
    print(chunk, end="", flush=True)
print()

print(f"\n📊 현재 대화 이력: {bot.get_history_count()}개 메시지")

💬 챗봇과 대화하기

👤 User: LoRA가 뭐야?
🤖 Bot: LoRA는 "Low-Rank Adaptation"의 약자로, 머신러닝 모델의 파라미터를 효율적으로 조정하기 위한 기법입니다. 주로 대규모 언어 모델이나 이미지 모델과 같은 복잡한 모델에서 사용됩니다.

LoRA의 핵심 아이디어는 전체 모델을 변경하는 대신, 모델의 가중치를 저차원 행렬로 근사하여 필요한 파라미터의 수를 줄이는 것입니다. 이렇게 하면 메모리 사용량과 계산 비용을 낮추면서도 모델의 성능을 유지할 수 있습니다.

### 예시
예를 들어, 대규모 언어 모델이 있다고 가정해봅시다. 이 모델을 새로운 데이터에 맞게 조정하려면 많은 파라미터를 업데이트해야 합니다. 하지만 LoRA를 사용하면, 전체 모델이 아닌 작은 저차원 행렬만 업데이트하여 필요한 정보를 추가할 수 있습니다. 

이렇게 하면 훈련 시간이 단축되고, 리소스도 절약할 수 있습니다. 

LoRA는 특히 리소스가 제한된 환경에서 큰 효과를 발휘합니다.

──────────────────────────────────────────────────

👤 User: 그거랑 QLoRA는 뭐가 달라?
🤖 Bot: LoRA와 QLoRA는 둘 다 모델의 파라미터 조정을 위한 기법이지만, QLoRA는 LoRA의 확장된 형태로, 양자화(Quantization) 기술을 결합한 것입니다.

### LoRA
- **기본 아이디어**: 모델의 가중치를 저차원 행렬로 근사하여 업데이트하는 방법.
- **장점**: 메모리와 계산 비용을 줄이면서도 성능을 유지할 수 있음.

### QLoRA
- **기본 아이디어**: LoRA의 아이디어를 기반으로 하면서, 추가로 양자화 기법을 사용하여 모델의 가중치를 더 작은 비트수로 표현합니다.
- **양자화(Quantization)**: 일반적으로 32비트 부동소수점 수로 표현되던 가중치를 4비트나 8비트처럼 더 작은 비트 수로 줄여서 메모리 사용량을 크게 감소시킵니다.
- **장점**: 메모리 사용량을 더욱 줄이면서도 모

In [16]:
# 대화 맥락 유지 확인
print("=" * 60)
print("🔄 대화 맥락 유지 확인")
print("=" * 60)

print("\n👤 User: 앞에서 설명해준 내용을 요약해줘.")
print(f"🤖 Bot: {bot.chat('앞에서 설명해준 내용을 요약해줘.')}")

print(f"\n📊 대화 이력: {bot.get_history_count()}개 메시지")
print("\n✅ 이전 대화 맥락을 기억하고 요약할 수 있습니다!")

# 대화 초기화
bot.reset()

🔄 대화 맥락 유지 확인

👤 User: 앞에서 설명해준 내용을 요약해줘.
🤖 Bot: 물론입니다! 다음은 요약입니다:

1. **LoRA (Low-Rank Adaptation)**:
   - 대규모 모델의 파라미터를 저차원 행렬로 근사하여 메모리와 계산 비용을 줄이는 기법.

2. **QLoRA**:
   - LoRA에 양자화(Quantization)를 결합하여 모델 가중치를 더 작은 비트수로 표현, 메모리 사용량을 더욱 줄임.

3. **8GB GPU에서 가능한 모델**:
   - BERT-base, GPT-2와 같은 중간 규모 모델은 학습 가능.
   - 배치 크기, 입력 데이터 크기, 훈련 방법에 따라 달라질 수 있음.

이 요약은 LoRA와 QLoRA의 차이점, 그리고 8GB GPU에서 학습할 수 있는 모델의 가능성을 간단히 정리한 것입니다.

📊 대화 이력: 8개 메시지

✅ 이전 대화 맥락을 기억하고 요약할 수 있습니다!
🔄 대화 이력이 초기화되었습니다.


In [17]:
# 🎮 인터랙티브 챗봇 (직접 대화해보기)
import sys

print("=" * 60)
print("🎮 인터랙티브 챗봇 — 직접 대화해보세요!")
print("=" * 60)
print("💡 'quit' 또는 'q'를 입력하면 종료합니다.")
print("💡 'reset'을 입력하면 대화 이력을 초기화합니다.")
print("💡 'history'를 입력하면 현재 대화 이력을 확인합니다.")
print()

# 위에서 만든 bot을 그대로 사용 (이력 초기화)
bot.reset()

while True:
    user_input = input("\n👤 You: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() in ['quit', 'q', '종료']:
        print("\n👋 대화를 종료합니다.")
        print(f"📊 총 대화 이력: {bot.get_history_count()}개 메시지")
        break
    
    if user_input.lower() == 'reset':
        bot.reset()
        continue
    
    if user_input.lower() == 'history':
        print(f"\n📊 현재 대화 이력: {bot.get_history_count()}개 메시지")
        for i, msg in enumerate(bot.history):
            role = "👤" if isinstance(msg, HumanMessage) else "🤖"
            content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
            print(f"   {role} [{i}] {content}")
        sys.stdout.flush()
        continue
    
    # 일반 응답 (Jupyter input() 버퍼링 문제 방지)
    response = bot.chat(user_input)
    print(f"🤖 Bot: {response}")
    sys.stdout.flush()


🎮 인터랙티브 챗봇 — 직접 대화해보세요!
💡 'quit' 또는 'q'를 입력하면 종료합니다.
💡 'reset'을 입력하면 대화 이력을 초기화합니다.
💡 'history'를 입력하면 현재 대화 이력을 확인합니다.

🔄 대화 이력이 초기화되었습니다.
🤖 Bot: 안녕하세요! 저는 AI와 머신러닝에 대해 배우고 싶어하는 여러분을 돕기 위해 여기 있는 튜터입니다. 기초부터 시작해서 점차 심화된 내용까지 함께 공부할 수 있도록 도와드릴게요. 질문이 있으면 언제든지 물어보세요!

👋 대화를 종료합니다.
📊 총 대화 이력: 2개 메시지


---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| LangChain 구조 | langchain-core, langchain-openai 등 모듈화된 구조 |
| ChatModel | ChatOpenAI로 모델 생성, invoke/stream/batch 지원 |
| PromptTemplate | ChatPromptTemplate으로 재사용 가능한 프롬프트 관리 |
| LCEL | `\|` 연산자로 prompt → model → parser 파이프라인 구성 |
| Output Parser | StrOutputParser, JsonOutputParser 등으로 출력 구조화 |
| Memory | MessagesPlaceholder + 대화 이력 리스트로 맥락 유지 |
| 챗봇 | 위 요소들을 조합하여 대화형 AI 구현 |

### LCEL 핵심 정리

```python
# 기본 체인
chain = prompt | model | parser

# 실행 방법
chain.invoke({...})   # 단일 실행
chain.stream({...})   # 스트리밍
chain.batch([...])    # 배치 처리
```

### 다음 파트 예고

- 🔜 Part 2: 모델 서빙(Ollama, Transformers) & RAG 파이프라인
- 🔜 로컬 모델 실행과 벡터 DB 활용을 배웁니다!

---

### 💡 실습 과제

1. SimpleChatbot 클래스를 활용하여 자신만의 전문 분야 챗봇을 만들어보세요.
   - 예: 요리 챗봇, 여행 가이드, 코딩 멘토 등
2. LCEL 파이프라인을 확장하여 '질문 → 답변 → 영어 번역' 3단계 체인을 만들어보세요.
3. JsonOutputParser를 활용하여 뉴스 기사에서 '제목, 날짜, 핵심 키워드, 요약'을 추출하는 체인을 만들어보세요.
4. (선택) ConversationSummaryMemory 방식을 구현해보세요: 5턴 이상의 오래된 대화를 요약하여 저장.

---